# χ4 Feasibility Probe — dynamic heterogeneity in iSCAT

**Standalone.** Computes χ4(τ) for the whole cell and checks for a peak at τ* — the
signature of **dynamic heterogeneity** (cooperative fast/slow switching, e.g. LLPS),
which the ACF / (γ,α) cannot see.

χ4(τ)=N·Var_t[Q(t,τ)], Q = spatial-mean overlap of z-scored fluctuations (per-frame
common-mode global flicker removed). A TIME-SHUFFLE control gives the noise floor.
Outcomes: **peak** (worth pursuing) / **no peak** / **SNR-starved** (honest degradation).

**Paths & preprocessing mirror `iscors_real_runner.ipynb` exactly** (same ZIP_PATH,
binning, flat-field, BG removal → identical `video_proc`).


In [ ]:
# ── setup: clone repo (for utils/gpu_chi4.py) + deps ─────────────────────
import os, subprocess, sys
REPO='https://github.com/breezy90126/iscors-net.git'
BRANCH='claude/brave-ramanujan-33eps3'
REPO_DIR='/content/iscors-net'
try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
except Exception: pass
if os.path.isdir(REPO_DIR):
    subprocess.run(['git','-C',REPO_DIR,'fetch','origin'],check=False)
    subprocess.run(['git','-C',REPO_DIR,'checkout',BRANCH],check=False)
    subprocess.run(['git','-C',REPO_DIR,'pull','origin',BRANCH],check=False)
else:
    subprocess.run(['git','clone','--branch',BRANCH,REPO,REPO_DIR],check=False)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
subprocess.run(['pip','install','-q','tifffile','scipy'],check=False)
print('setup done:', os.getcwd())


In [ ]:
# ── config — SAME paths as iscors_real_runner.ipynb (p2-config) ──────────
import os, numpy as np
ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'   # raw video zip
EXTRACT_DIR = '/content/real_data'                                  # local extraction
SAVE_DIR    = '/content/drive/MyDrive/iscors_test'                  # results dir
VIDEO_FNAME = 'COBRI_rarw_video.tif'
N_FRAMES    = 2000        # χ4 wants as many frames as possible
BIN_FACTOR  = 2
CHUNK_SIZE  = 100
MIN_CV      = 0.005
TAUS        = (1,2,3,4,6,8,12,16,24,32,48,64,96,128,192,256)  # denser τ → cleaner peak
os.makedirs(EXTRACT_DIR, exist_ok=True); os.makedirs(SAVE_DIR, exist_ok=True)
print('ZIP_PATH :', ZIP_PATH)
print('TAUS     :', TAUS)


In [ ]:
# ── load + preprocess (identical pipeline to p2-preprocess) ──────────────
import zipfile, tifffile
from scipy.ndimage import gaussian_filter
def _find(root, name):
    for dp, _, fs in os.walk(root):
        if name in fs: return os.path.join(dp, name)
    return None
VIDEO_PATH = _find(EXTRACT_DIR, VIDEO_FNAME)
if not VIDEO_PATH:
    print(f'Extracting {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH,'r') as zf: zf.extractall(EXTRACT_DIR)
    VIDEO_PATH = _find(EXTRACT_DIR, VIDEO_FNAME)
assert VIDEO_PATH, f'{VIDEO_FNAME} not found under {EXTRACT_DIR}'
print('Video:', VIDEO_PATH)

frame0 = tifffile.imread(VIDEO_PATH, key=0).astype(np.float32)
H0, W0 = frame0.shape; Hb, Wb = H0//BIN_FACTOR, W0//BIN_FACTOR
with tifffile.TiffFile(VIDEO_PATH) as tf: total = len(tf.pages)
n = min(N_FRAMES, total)
print(f'Loading {n}/{total} frames, {H0}×{W0} → {Hb}×{Wb} (bin {BIN_FACTOR}) ...')
video_raw = np.empty((n, Hb, Wb), np.float32)
for s in range(0, n, CHUNK_SIZE):
    e = min(s+CHUNK_SIZE, n)
    ch = tifffile.imread(VIDEO_PATH, key=range(s, e)).astype(np.float32)
    video_raw[s:e] = ch.reshape(e-s, Hb, BIN_FACTOR, Wb, BIN_FACTOR).mean((2,4))
    del ch
# Step 1: flat-field ; Step 2: per-frame Gaussian BG removal (same as main)
median_xy = np.median(video_raw, axis=0)
video_ff  = video_raw / (median_xy[None] + 1e-10)
video_proc = np.empty_like(video_ff)
for t in range(len(video_ff)):
    bg = gaussian_filter(video_ff[t], sigma=4)
    video_proc[t] = video_ff[t] / (bg + 1e-10)
print(f'video_proc: {video_proc.shape}  mean={video_proc.mean():.4f} (≈1 ✓)')


In [ ]:
# ── compute χ4(τ) + time-shuffle control + verdict ───────────────────────
import importlib, utils.gpu_chi4 as _c4; importlib.reload(_c4)
from utils.gpu_chi4 import chi4_probe
out = chi4_probe(video_proc, TAUS, min_cv=MIN_CV, verbose=True)
print()
print('τ      :', out['taus'].astype(int))
print('χ4     :', np.round(out['chi4'],1))
print('χ4 shuf:', np.round(out['chi4_shuffled'],1))
print('relax  :', np.round(out['relax'],3), ' (should decay with τ)')


In [ ]:
# ── plot + record ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2, figsize=(12,4.5))
ax[0].semilogx(out['taus'], out['chi4'], 'o-', label='χ4 (real)')
ax[0].semilogx(out['taus'], out['chi4_shuffled'], 's--', color='gray', label='χ4 (time-shuffled floor)')
if out['resolved']: ax[0].axvline(out['tau_star'], color='r', ls=':', label=f"τ*={out['tau_star']:.0f}")
ax[0].set_xlabel('τ (lag)'); ax[0].set_ylabel('χ4(τ)'); ax[0].legend(fontsize=8)
ax[0].set_title(f"χ4 (N={out['N']} px)  peak={out['peak']:.1f}  {out['snr']:.1f}× floor")
ax[1].semilogx(out['taus'], out['relax'], 'o-'); ax[1].set_xlabel('τ (lag)')
ax[1].set_ylabel('⟨Q(τ)⟩'); ax[1].set_title('overlap relaxation (sanity: should decay)')
verdict = ('RESOLVED: dynamic heterogeneity, τ*=%.0f → χ4 worth pursuing'%out['tau_star']
           if out['resolved'] else
           'NOT resolved: no peak above noise floor → homogeneous/static/SNR-starved')
fig.suptitle('χ4 feasibility probe — '+verdict, fontsize=12)
plt.tight_layout(); plt.show()
fig.savefig(os.path.join(SAVE_DIR,'chi4_probe.png'), dpi=120, bbox_inches='tight')
with open(os.path.join(SAVE_DIR,'chi4_probe.txt'),'w') as f:
    f.write('=== chi4 feasibility probe ===\n')
    f.write(f"N_cell={out['N']}  peak={out['peak']:.2f}  shuffle_floor={out['shuffle_floor']:.2f}  snr={out['snr']:.2f}\n")
    f.write(f"tau_star={out['tau_star']}  resolved={out['resolved']}\n")
    f.write('VERDICT: '+verdict+'\n')
print('VERDICT:', verdict); print('saved →', SAVE_DIR)


## How to read it
- **Clear peak at τ*, well above the shuffled (gray) floor** → dynamic heterogeneity is
  real and resolvable; τ* is its timescale, peak height ≈ size of cooperative regions →
  χ4 worth developing (incl. an ML estimator — classical χ4 is noise-starved).
- **χ4 ≈ shuffled floor / no peak** → no resolvable dynamic heterogeneity (homogeneous,
  static, or SNR-starved). Honest degradation — like STICS sub-PSF.
- **`relax` not decaying** → preprocessing issue (common-mode/drift); fix before trusting χ4.
